# Week 6 Recap Lab: Pandas + SQLAlchemy, DuckDB and Polars

Checkpoint preparation lab — revised version - Sébastien Vouters

This notebook was created and runned on MacOS

In [97]:
from pathlib import Path
import pandas as pd
import pyodbc
import duckdb
import polars as pl

from sqlalchemy import create_engine, text
from urllib.parse import quote_plus

In [98]:
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
RAW = PROJECT_ROOT / 'data' / 'raw'
OUTPUT = PROJECT_ROOT / 'data' / 'output'
CURATED = PROJECT_ROOT / "data" / "curated"

OUTPUT.mkdir(parents=True, exist_ok=True)
CURATED.mkdir(parents=True, exist_ok=True)

print('Project root:', PROJECT_ROOT)

Project root: /Users/seb_el_famoso/Documents/Brights/Week 6 - Python Advanced/week6_starter


## Concept warm-ups

### Warm-up 1 — SQLAlchemy connection and simple read_sql

In [132]:
server = r"localhost"  # change if needed

connection_string = 'DRIVER={ODBC Driver 17 for SQL Server};SERVER=localhost;DATABASE=W6RecapDB;UID=SA;PWD=Bo723141'

connection_url = "mssql+pyodbc:///?odbc_connect=" + quote_plus(connection_string)
engine = create_engine(connection_url, fast_executemany=True)
print(engine)

Engine(mssql+pyodbc:///?odbc_connect=DRIVER%3D%7BODBC+Driver+17+for+SQL+Server%7D%3BSERVER%3Dlocalhost%3BDATABASE%3DW6RecapDB%3BUID%3DSA%3BPWD%3DBo723141)


In [133]:
with engine.connect() as conn:
    display(pd.read_sql("SELECT TOP (3) * FROM dbo.Customers ORDER BY customer_id", conn))

,customer_id,customer_name,country,segment
0,C001,Aalto Bikes,Finland,Retail
1,C002,Data School,Finland,Education
2,C003,Nordic Mobility,Sweden,Enterprise


### Warm-up 2 — Pandas concat with source labels

In [134]:
jan_demo = pd.DataFrame({"order_id": [1, 2], "amount": [100, 200]})
feb_demo = pd.DataFrame({"order_id": [3, 4], "amount": [300, 400]})
jan_demo["source_month"] = "Jan"
feb_demo["source_month"] = "Feb"
concat_demo = pd.concat([jan_demo, feb_demo], ignore_index=True)
display(concat_demo)

,order_id,amount,source_month
0,1,100,Jan
1,2,200,Jan
2,3,300,Feb
3,4,400,Feb


### Warm-up 3 — Pandas numeric_from_text helper

In [102]:
def numeric_from_text(series):
    return pd.to_numeric(
        series.astype("string")
            .str.replace("€", "", regex=False)
            .str.replace(" ", "", regex=False)
            .str.replace(",", ".", regex=False),
        errors="coerce"
    )

### Warm-up 4 — Pandas merge with reference data

In [103]:
orders_demo = pd.DataFrame({"order_id": [1, 2, 3], "customer_id": ["C001", "C002", "C999"]})
customers_demo = pd.DataFrame({"customer_id": ["C001", "C002"], "customer_name": ["Aalto", "Data School"]})
merged_demo = orders_demo.merge(customers_demo, how="left", on="customer_id")
display(merged_demo)

,order_id,customer_id,customer_name
0,1,C001,Aalto
1,2,C002,Data School
2,3,C999,NaN


### Warm-up 5 — DuckDB read CSV directly

In [104]:
duck = duckdb.connect()
display(duck.sql(f"SELECT COUNT(*) AS row_count FROM read_csv_auto('{RAW /
'sales_orders_raw.csv'}')").df())

,row_count
0,20


### Warm-up 6 — DuckDB GROUP BY over a CSV

In [105]:
display(duck.sql(f"""
    SELECT status_text, COUNT(*) AS row_count
    FROM read_csv_auto('{RAW / "sales_orders_raw.csv"}')
    GROUP BY status_text
    ORDER BY row_count DESC
""").df())

,status_text,row_count
0,Waiting,7
1,Open,6
2,Paid,5
3,Cancelled,2


### Warm-up 7 — DuckDB COPY TO Parquet

In [106]:
duck.sql(f"""
    COPY (
        SELECT status_text, COUNT(*) AS row_count
        FROM read_csv_auto('{RAW / "sales_orders_raw.csv"}')
        GROUP BY status_text
    )
    TO '{OUTPUT / "warmup_status_counts.parquet"}'
    (FORMAT PARQUET)
""")

### Warm-up 8 — Polars select, filter and sort

In [107]:
mini = pl.DataFrame({
    "customer": [" Aalto ", "Data School", "Cloud Ltd"],
    "country": ["Finland", "Finland", "Sweden"],
    "amount_text": ["100", "250", "bad"]
})
mini.filter(pl.col("country") == "Finland").sort("customer")

customer,country,amount_text
str,str,str
""" Aalto ""","""Finland""","""100"""
"""Data School""","""Finland""","""250"""


### Warm-up 9 — Polars with_columns, string functions and numeric cast

In [108]:
mini_clean = mini.with_columns(
    pl.col("customer").str.strip_chars().str.to_uppercase().alias("customer_clean"),
    pl.col("amount_text").cast(pl.Float64, strict=False).alias("amount")
)
mini_clean

customer,country,amount_text,customer_clean,amount
str,str,str,str,f64
""" Aalto ""","""Finland""","""100""","""AALTO""",100.0
"""Data School""","""Finland""","""250""","""DATA SCHOOL""",250.0
"""Cloud Ltd""","""Sweden""","""bad""","""CLOUD LTD""",null


### Warm-up 10 — Polars group_by and aggregation

In [109]:
mini_summary = (
    mini_clean
    .group_by("country")
    .agg(
        pl.len().alias("row_count"),
        pl.col("amount").is_not_null().sum().alias("valid_amount_rows"),
        pl.sum("amount").alias("total_amount")
    )
    .sort("country")
)
mini_summary

country,row_count,valid_amount_rows,total_amount
str,u32,u32,f64
"""Finland""",2,2,350.0
"""Sweden""",1,0,0.0


### Warm-up 11 — Polars dependent columns need another with_columns

In [110]:
# Rule: if a new column depends on another new column, use another with_columns().
price_demo = pl.DataFrame({
    "product": ["Book", "Monitor", "Cable"],
    "quantity_text": ["2", "1", "bad"],
    "unit_price_text": ["10,00", "250,00", "5,50"]
})
price_demo_clean = (
    price_demo
    .with_columns(
        pl.col("quantity_text").cast(pl.Int64, strict=False).alias("quantity"),
        pl.col("unit_price_text").str.replace_all(",", ".").cast(pl.Float64,
        strict=False).alias("unit_price")
    )
    .with_columns(
        # This uses quantity and unit_price created above.
        (pl.col("quantity") * pl.col("unit_price")).alias("line_amount")
    )
)
price_demo

product,quantity_text,unit_price_text
str,str,str
"""Book""","""2""","""10,00"""
"""Monitor""","""1""","""250,00"""
"""Cable""","""bad""","""5,50"""


## Required base tasks

### Part 1 — Pandas and SQLAlchemy

#### Task 1 — Read SQL Server tables into Pandas

In [111]:
with engine.connect() as conn:
    customers_df = pd.read_sql("SELECT * FROM dbo.Customers ORDER BY customer_id", conn)
    products_df = pd.read_sql("SELECT * FROM dbo.Products ORDER BY product_id", conn)
    orders_jan_df = pd.read_sql("SELECT * FROM dbo.Orders_Jan ORDER BY order_id", conn)
    orders_feb_df = pd.read_sql("SELECT * FROM dbo.Orders_Feb ORDER BY order_id", conn)

table_list = [customers_df, products_df, orders_jan_df, orders_feb_df]
for df in table_list :
    print(df.shape)
    print(df.dtypes)
    display(df.head(3))

(6, 4)
customer_id      str
customer_name    str
country          str
segment          str
dtype: object


,customer_id,customer_name,country,segment
0,C001,Aalto Bikes,Finland,Retail
1,C002,Data School,Finland,Education
2,C003,Nordic Mobility,Sweden,Enterprise


(6, 4)
product_id            str
product_name          str
category              str
standard_price    float64
dtype: object


,product_id,product_name,category,standard_price
0,P001,City Bike,Bike,650.0
1,P002,Helmet,Accessory,45.0
2,P003,Lock,Accessory,30.0


(8, 8)
order_id            int64
order_date_text       str
customer_id_text      str
product_id_text       str
quantity_text         str
unit_price_text       str
discount_text         str
status_text           str
dtype: object


,order_id,order_date_text,customer_id_text,product_id_text,quantity_text,unit_price_text,discount_text,status_text
0,1001,2026-01-05,C001,P001,2,650.00,0.00,paid
1,1002,2026-01-06,c002,P006,1,"€250,00","€0,00",Open
2,1003,2026-01-09,C003,P002,5,45.00,5.00,PAID


(8, 8)
order_id            int64
order_date_text       str
customer_id_text      str
product_id_text       str
quantity_text         str
unit_price_text       str
discount_text         str
status_text           str
dtype: object


,order_id,order_date_text,customer_id_text,product_id_text,quantity_text,unit_price_text,discount_text,status_text
0,2001,2026-02-03,C001,P004,2,120.00,0.00,Open
1,2002,2026-02-05,C002,P001,1,650.00,25.00,Paid
2,2003,2026-02-08,C003,P005,1,"€1 450,00","€100,00",paid


#### Task 2 — Add source month and concatenate orders

In [112]:
orders_jan_df['source_month']='Jan'
orders_feb_df['source_month']='Feb'

orders_all = pd.concat([orders_jan_df, orders_feb_df], ignore_index=True)
print('row count :', len(orders_all))
display(orders_all.head(10))

row count : 16


,order_id,order_date_text,customer_id_text,product_id_text,quantity_text,unit_price_text,discount_text,status_text,source_month
0,1001,2026-01-05,C001,P001,2,650.00,0.00,paid,Jan
1,1002,2026-01-06,c002,P006,1,"€250,00","€0,00",Open,Jan
2,1003,2026-01-09,C003,P002,5,45.00,5.00,PAID,Jan
3,1004,2026-01-10,C999,P004,1,120.00,0.00,Paid,Jan
4,1005,bad-date,C001,P005,1,1450.00,50.00,Waiting,Jan
5,1006,2026-01-15,C004,P999,2,30.00,0.00,Paid,Jan
6,1007,2026-01-18,C005,P003,bad,30.00,0.00,Cancelled,Jan
7,1008,2026-01-22,C006,P006,3,250.00,25.00,Pending,Jan
8,2001,2026-02-03,C001,P004,2,120.00,0.00,Open,Feb
9,2002,2026-02-05,C002,P001,1,650.00,25.00,Paid,Feb


#### Task 3 — Clean IDs, dates, status and numeric text columns

In [113]:
status_clean = {
    'PAID':'Paid',
    'OPEN':'Open',
    'CANCELLED':'Cancelled',
    'CANCELED':'Cancelled',
    'PENDING':'Pending',
    'WAITING':'Pending'
}

clean_orders_df = orders_all.copy()
clean_orders_df['customer_id'] = clean_orders_df['customer_id_text'].str.strip().str.replace(' ','').str.upper()
clean_orders_df['product_id'] = clean_orders_df['product_id_text'].str.strip().str.replace(' ','').str.upper()

clean_orders_df['order_date'] = pd.to_datetime(clean_orders_df['order_date_text'], errors='coerce')

clean_orders_df['status_clean'] = clean_orders_df['status_text'].str.strip().str.upper().map(status_clean)

clean_orders_df['quantity'] = clean_orders_df['quantity_text'].str.replace("€", "", regex=False).str.replace(" ", "", regex=False).str.replace(",", ".", regex=False)
clean_orders_df['unit_price'] = clean_orders_df['unit_price_text'].str.replace("€", "", regex=False).str.replace(" ", "", regex=False).str.replace(",", ".", regex=False)
clean_orders_df['discount'] = clean_orders_df['discount_text'].str.replace("€", "", regex=False).str.replace(" ", "", regex=False).str.replace(",", ".", regex=False)
clean_orders_df['quantity'] = pd.to_numeric(clean_orders_df['quantity'], errors='coerce')
clean_orders_df['unit_price'] = pd.to_numeric(clean_orders_df['unit_price'], errors='coerce')
clean_orders_df['discount'] = pd.to_numeric(clean_orders_df['discount'], errors='coerce')

display(clean_orders_df[['order_id', 'customer_id', 'product_id', 'order_date', 'quantity', 'unit_price', 'discount', 'status_clean']])

,order_id,customer_id,product_id,order_date,quantity,unit_price,discount,status_clean
0,1001,C001,P001,2026-01-05,2.0,650.0,0.0,Paid
1,1002,C002,P006,2026-01-06,1.0,250.0,0.0,Open
2,1003,C003,P002,2026-01-09,5.0,45.0,5.0,Paid
3,1004,C999,P004,2026-01-10,1.0,120.0,0.0,Paid
4,1005,C001,P005,NaT,1.0,1450.0,50.0,Pending
5,1006,C004,P999,2026-01-15,2.0,30.0,0.0,Paid
6,1007,C005,P003,2026-01-18,NaN,30.0,0.0,Cancelled
7,1008,C006,P006,2026-01-22,3.0,250.0,25.0,Pending
8,2001,C001,P004,2026-02-03,2.0,120.0,0.0,Open
9,2002,C002,P001,2026-02-05,1.0,650.0,25.0,Paid


#### Task 4 — Merge cleaned orders with customers and products

In [114]:
orders_enriched = clean_orders_df.merge(customers_df, on='customer_id', how='left')
orders_enriched = orders_enriched.merge(products_df, on='product_id', how='left')
orders_enriched

,order_id,order_date_text,customer_id_text,product_id_text,quantity_text,unit_price_text,discount_text,status_text,source_month,customer_id,...,status_clean,quantity,unit_price,discount,customer_name,country,segment,product_name,category,standard_price
0,1001,2026-01-05,C001,P001,2,650.00,0.00,paid,Jan,C001,...,Paid,2.0,650.0,0.0,Aalto Bikes,Finland,Retail,City Bike,Bike,650.0
1,1002,2026-01-06,c002,P006,1,"€250,00","€0,00",Open,Jan,C002,...,Open,1.0,250.0,0.0,Data School,Finland,Education,Training Session,Training,250.0
2,1003,2026-01-09,C003,P002,5,45.00,5.00,PAID,Jan,C003,...,Paid,5.0,45.0,5.0,Nordic Mobility,Sweden,Enterprise,Helmet,Accessory,45.0
3,1004,2026-01-10,C999,P004,1,120.00,0.00,Paid,Jan,C999,...,Paid,1.0,120.0,0.0,NaN,NaN,NaN,Service Package,Service,120.0
4,1005,bad-date,C001,P005,1,1450.00,50.00,Waiting,Jan,C001,...,Pending,1.0,1450.0,50.0,Aalto Bikes,Finland,Retail,E-Bike,Bike,1450.0
5,1006,2026-01-15,C004,P999,2,30.00,0.00,Paid,Jan,C004,...,Paid,2.0,30.0,0.0,Cloud Rentals,Norway,Enterprise,NaN,NaN,NaN
6,1007,2026-01-18,C005,P003,bad,30.00,0.00,Cancelled,Jan,C005,...,Cancelled,NaN,30.0,0.0,Baltic Bikes,Estonia,Retail,Lock,Accessory,30.0
7,1008,2026-01-22,C006,P006,3,250.00,25.00,Pending,Jan,C006,...,Pending,3.0,250.0,25.0,Learning Hub,Denmark,Education,Training Session,Training,250.0
8,2001,2026-02-03,C001,P004,2,120.00,0.00,Open,Feb,C001,...,Open,2.0,120.0,0.0,Aalto Bikes,Finland,Retail,Service Package,Service,120.0
9,2002,2026-02-05,C002,P001,1,650.00,25.00,Paid,Feb,C002,...,Paid,1.0,650.0,25.0,Data School,Finland,Education,City Bike,Bike,650.0


In [115]:
unmatched_orders = orders_enriched[orders_enriched['customer_name'].isna() | orders_enriched['product_name'].isna()]
unmatched_orders

,order_id,order_date_text,customer_id_text,product_id_text,quantity_text,unit_price_text,discount_text,status_text,source_month,customer_id,...,status_clean,quantity,unit_price,discount,customer_name,country,segment,product_name,category,standard_price
3,1004,2026-01-10,C999,P004,1,120.00,0.00,Paid,Jan,C999,...,Paid,1.0,120.0,0.0,NaN,NaN,NaN,Service Package,Service,120.0
5,1006,2026-01-15,C004,P999,2,30.00,0.00,Paid,Jan,C004,...,Paid,2.0,30.0,0.0,Cloud Rentals,Norway,Enterprise,NaN,NaN,NaN


#### Task 5 — Calculate line amount and classify amount band

In [116]:
orders_enriched['line_amount'] = orders_enriched['quantity'] * (orders_enriched['unit_price'] - orders_enriched['discount'])
orders_enriched

,order_id,order_date_text,customer_id_text,product_id_text,quantity_text,unit_price_text,discount_text,status_text,source_month,customer_id,...,quantity,unit_price,discount,customer_name,country,segment,product_name,category,standard_price,line_amount
0,1001,2026-01-05,C001,P001,2,650.00,0.00,paid,Jan,C001,...,2.0,650.0,0.0,Aalto Bikes,Finland,Retail,City Bike,Bike,650.0,1300.0
1,1002,2026-01-06,c002,P006,1,"€250,00","€0,00",Open,Jan,C002,...,1.0,250.0,0.0,Data School,Finland,Education,Training Session,Training,250.0,250.0
2,1003,2026-01-09,C003,P002,5,45.00,5.00,PAID,Jan,C003,...,5.0,45.0,5.0,Nordic Mobility,Sweden,Enterprise,Helmet,Accessory,45.0,200.0
3,1004,2026-01-10,C999,P004,1,120.00,0.00,Paid,Jan,C999,...,1.0,120.0,0.0,NaN,NaN,NaN,Service Package,Service,120.0,120.0
4,1005,bad-date,C001,P005,1,1450.00,50.00,Waiting,Jan,C001,...,1.0,1450.0,50.0,Aalto Bikes,Finland,Retail,E-Bike,Bike,1450.0,1400.0
5,1006,2026-01-15,C004,P999,2,30.00,0.00,Paid,Jan,C004,...,2.0,30.0,0.0,Cloud Rentals,Norway,Enterprise,NaN,NaN,NaN,60.0
6,1007,2026-01-18,C005,P003,bad,30.00,0.00,Cancelled,Jan,C005,...,NaN,30.0,0.0,Baltic Bikes,Estonia,Retail,Lock,Accessory,30.0,NaN
7,1008,2026-01-22,C006,P006,3,250.00,25.00,Pending,Jan,C006,...,3.0,250.0,25.0,Learning Hub,Denmark,Education,Training Session,Training,250.0,675.0
8,2001,2026-02-03,C001,P004,2,120.00,0.00,Open,Feb,C001,...,2.0,120.0,0.0,Aalto Bikes,Finland,Retail,Service Package,Service,120.0,240.0
9,2002,2026-02-05,C002,P001,1,650.00,25.00,Paid,Feb,C002,...,1.0,650.0,25.0,Data School,Finland,Education,City Bike,Bike,650.0,625.0


In [117]:
def classify_amount(row) :
    amount = row['line_amount']
    if amount >= 1000 :
        return 'High'
    elif amount >= 250 :
        return 'Medium'
    elif amount >= 0 :
        return 'Low'
    else :
        return 'Unknown'
    
orders_enriched['amount_band'] = orders_enriched.apply(classify_amount, axis=1)
orders_enriched

,order_id,order_date_text,customer_id_text,product_id_text,quantity_text,unit_price_text,discount_text,status_text,source_month,customer_id,...,unit_price,discount,customer_name,country,segment,product_name,category,standard_price,line_amount,amount_band
0,1001,2026-01-05,C001,P001,2,650.00,0.00,paid,Jan,C001,...,650.0,0.0,Aalto Bikes,Finland,Retail,City Bike,Bike,650.0,1300.0,High
1,1002,2026-01-06,c002,P006,1,"€250,00","€0,00",Open,Jan,C002,...,250.0,0.0,Data School,Finland,Education,Training Session,Training,250.0,250.0,Medium
2,1003,2026-01-09,C003,P002,5,45.00,5.00,PAID,Jan,C003,...,45.0,5.0,Nordic Mobility,Sweden,Enterprise,Helmet,Accessory,45.0,200.0,Low
3,1004,2026-01-10,C999,P004,1,120.00,0.00,Paid,Jan,C999,...,120.0,0.0,NaN,NaN,NaN,Service Package,Service,120.0,120.0,Low
4,1005,bad-date,C001,P005,1,1450.00,50.00,Waiting,Jan,C001,...,1450.0,50.0,Aalto Bikes,Finland,Retail,E-Bike,Bike,1450.0,1400.0,High
5,1006,2026-01-15,C004,P999,2,30.00,0.00,Paid,Jan,C004,...,30.0,0.0,Cloud Rentals,Norway,Enterprise,NaN,NaN,NaN,60.0,Low
6,1007,2026-01-18,C005,P003,bad,30.00,0.00,Cancelled,Jan,C005,...,30.0,0.0,Baltic Bikes,Estonia,Retail,Lock,Accessory,30.0,NaN,Unknown
7,1008,2026-01-22,C006,P006,3,250.00,25.00,Pending,Jan,C006,...,250.0,25.0,Learning Hub,Denmark,Education,Training Session,Training,250.0,675.0,Medium
8,2001,2026-02-03,C001,P004,2,120.00,0.00,Open,Feb,C001,...,120.0,0.0,Aalto Bikes,Finland,Retail,Service Package,Service,120.0,240.0,Low
9,2002,2026-02-05,C002,P001,1,650.00,25.00,Paid,Feb,C002,...,650.0,25.0,Data School,Finland,Education,City Bike,Bike,650.0,625.0,Medium


#### Task 6 — Create country/status summary and write it to SQL Server

In [118]:
valid_orders = orders_enriched[~(orders_enriched['customer_name'].isna() | orders_enriched['product_name'].isna() | orders_enriched['line_amount'].isna())]
valid_orders

,order_id,order_date_text,customer_id_text,product_id_text,quantity_text,unit_price_text,discount_text,status_text,source_month,customer_id,...,unit_price,discount,customer_name,country,segment,product_name,category,standard_price,line_amount,amount_band
0,1001,2026-01-05,C001,P001,2,650.00,0.00,paid,Jan,C001,...,650.0,0.0,Aalto Bikes,Finland,Retail,City Bike,Bike,650.0,1300.0,High
1,1002,2026-01-06,c002,P006,1,"€250,00","€0,00",Open,Jan,C002,...,250.0,0.0,Data School,Finland,Education,Training Session,Training,250.0,250.0,Medium
2,1003,2026-01-09,C003,P002,5,45.00,5.00,PAID,Jan,C003,...,45.0,5.0,Nordic Mobility,Sweden,Enterprise,Helmet,Accessory,45.0,200.0,Low
4,1005,bad-date,C001,P005,1,1450.00,50.00,Waiting,Jan,C001,...,1450.0,50.0,Aalto Bikes,Finland,Retail,E-Bike,Bike,1450.0,1400.0,High
7,1008,2026-01-22,C006,P006,3,250.00,25.00,Pending,Jan,C006,...,250.0,25.0,Learning Hub,Denmark,Education,Training Session,Training,250.0,675.0,Medium
8,2001,2026-02-03,C001,P004,2,120.00,0.00,Open,Feb,C001,...,120.0,0.0,Aalto Bikes,Finland,Retail,Service Package,Service,120.0,240.0,Low
9,2002,2026-02-05,C002,P001,1,650.00,25.00,Paid,Feb,C002,...,650.0,25.0,Data School,Finland,Education,City Bike,Bike,650.0,625.0,Medium
10,2003,2026-02-08,C003,P005,1,"€1 450,00","€100,00",paid,Feb,C003,...,1450.0,100.0,Nordic Mobility,Sweden,Enterprise,E-Bike,Bike,1450.0,1350.0,High
11,2004,2026-02-11,C004,P002,4,45.00,0.00,Cancelled,Feb,C004,...,45.0,0.0,Cloud Rentals,Norway,Enterprise,Helmet,Accessory,45.0,180.0,Low
12,2005,2026-02-14,C005,P003,10,30.00,-5.00,Paid,Feb,C005,...,30.0,-5.0,Baltic Bikes,Estonia,Retail,Lock,Accessory,30.0,350.0,Medium


In [120]:
country_status_summary = valid_orders.groupby(['country', 'status_clean']).agg(
    order_count=("order_id", "count"),
    total_quantity=("quantity", "sum"),
    total_line_amount=("line_amount", "sum")
).reset_index()
country_status_summary

,country,status_clean,order_count,total_quantity,total_line_amount
0,Denmark,Pending,1,3.0,675.0
1,Estonia,Paid,1,10.0,350.0
2,Finland,Open,3,3.0,490.0
3,Finland,Paid,3,4.0,3375.0
4,Finland,Pending,1,1.0,1400.0
5,Norway,Cancelled,1,4.0,180.0
6,Sweden,Paid,2,6.0,1550.0


In [123]:
country_status_summary.to_sql('RecapCountryStatusSummary', engine, schema='dbo', if_exists='replace', index=False)
country_status_summary_backup = pd.read_sql(text('SELECT * FROM dbo.RecapCountryStatusSummary'), engine)
country_status_summary_backup

,country,status_clean,order_count,total_quantity,total_line_amount
0,Denmark,Pending,1,3.0,675.0
1,Estonia,Paid,1,10.0,350.0
2,Finland,Open,3,3.0,490.0
3,Finland,Paid,3,4.0,3375.0
4,Finland,Pending,1,1.0,1400.0
5,Norway,Cancelled,1,4.0,180.0
6,Sweden,Paid,2,6.0,1550.0


### Part 2 — DuckDB

#### Task 7 — Count raw CSV rows by status

In [124]:
duck = duckdb.connect()
display(duck.sql(f"SELECT COUNT(*) AS row_count FROM read_csv_auto('{RAW /
'sales_orders_raw.csv'}')").df())

,row_count
0,20


In [131]:
duckdb_status_summary = duck.sql(f"""
    SELECT
        status_text,
        COUNT(*) AS row_count
    FROM read_csv_auto('{RAW / "sales_orders_raw.csv"}')
    GROUP BY status_text
    ORDER BY row_count DESC, status_text ASC
""").df()

display(duckdb_status_summary)

,status_text,row_count
0,Waiting,7
1,Open,6
2,Paid,5
3,Cancelled,2


#### Task 8 — Join raw sales orders to products and summarize by category